# Проект III: Интерактивная веб-карта
Бронина Виктория (ЦУиАГм-1).


## 0. Импорт библиотек и подготовка данных

### 0.1 Импорт библиотек

In [1]:
import folium
import geopandas as gpd
import networkx as nx
import osmnx as ox
from shapely.geometry import Polygon

### 0.2 Подготовка данных
Интерактивная карта будет содержать следующую информацию по району Южное Бутово г. Москвы: 
- типы землепользования района
- застройка района (с возможностью получения информации по типу здания - всплывающая подсказка)
- улично-дорожная сеть (исключительно пешеходная, так как в карте определяется пешеходная доступность)
- обслуживающие станции метрополитена
- доступность станций метрополитена (построение изохрон по графу пешеходной сети)

Так как часть необходимых данных уже использовалась мной ранее в предыдущих проектах, выгружаю их "чтением"" (файлы уже обработаны ранее):

In [2]:
area_border = gpd.read_file("../data/osm_border.geojson")
buildings = gpd.read_file("../data/osm_build.geojson")
landuse = gpd.read_file("../data/osm_landuse.geojson")

Остальные osm-данные необходимо выгрузить через osmnx:

In [3]:
# создаем граф пешеходной сети
G_walk = ox.graph_from_polygon(
    area_border.geometry.iloc[0], 
    network_type="walk"
)
# переводим в метры для дальнейшего расчета изохрон
G_walk_projected = ox.project_graph(G_walk)

In [4]:
# выгружаем станций метрополитена
tags_metro = {
    "railway": ["subway_entrance"]
}
metro_stations = ox.features_from_polygon(area_border.geometry.iloc[0], tags_metro)

## 1. Построение интерактивной карты

На карте  далее будут выстроены изохроны пешеходной доступности от станции метрополитена;

Согласно федеральным нормативам (СП 42.13330.2016 «Градостроительство») и региональным стандартам (ПП Москвы № 945-PP), максимальный нормативный радиус пешеходной доступности станций метрополитена составляет от 500 до 700 метров.

В среднем это эквивалентно 10–12 минутам спокойной ходьбы.


In [5]:
# НАСТРОЙКА РАСЧЕТНЫХ ПАРАМЕТРОВ
walk_speed = 4.5  # cкорость пешехода, км/ч
meters_per_minute = (walk_speed * 1000) / 60  # перевод заданной скорости в м/мин

iso_min = [15, 10, 5] # задаем интервалы изохрон в минутах (норматив - до 10 мин)
iso_colors = {5: "#2ca02c", 10: "#ff7f0e", 15: "#d62728"} # настраиваем цвета изохрон (по "светофору" нормативного времени)


# НАСТРОЙКА ЦВЕТОВ ЗЕМЛЕПОЛЬЗОВАНИЯ
landuse_colors = {
    "park": {"fill": "#a1d99b", "stroke": "#74c476", "label": "Парки и сады"},
    "garden": {"fill": "#a1d99b", "stroke": "#74c476", "label": "Парки и сады"},
    "grass": {"fill": "#a1d99b", "stroke": "#74c476", "label": "Парки и сады"},
    "forest": {"fill": "#a1d99b", "stroke": "#74c476", "label": "Парки и сады"},
    "residential": {"fill": "#ffeda0", "stroke": "#fed976", "label": "Жилая застройка"},
    "industrial": {"fill": "#d7b5d8", "stroke": "#df65b0", "label": "Промзоны"},
    "allotments": {"fill": "#e5f5e0", "stroke": "#a1d99b", "label": "Садоводства / Дачи"},
    "cemetery": {"fill": "#bcbddc", "stroke": "#9e9ac8", "label": "Кладбища"},
    "construction": {"fill": "#fee6ce", "stroke": "#fdae6b", "label": "Строительные площадки"},
    "default": {"fill": "#f0f0f0", "stroke": "#d9d9d9", "label": "Прочие территории"}
}


# СОЗДАНИЕ ИЗОХРОН
# Цикл для расчета времени прохождения по каждому ребру графа (в минутах)
for u, v, k, data in G_walk_projected.edges(data=True, keys=True):
    data["time"] = data["length"] / meters_per_minute

isochrone_polygons = {min_val: [] for min_val in iso_min}

# Начинаем "перебирать" станции метро и определять зоны доступности
if not metro_stations.empty:
    metro_points_proj = metro_stations.to_crs(G_walk_projected.graph["crs"]) # проецируем точки метро в ту же UTM проекцию, что и граф дорог
    center_nodes = ox.nearest_nodes(
        G_walk_projected, X=metro_points_proj.geometry.x, Y=metro_points_proj.geometry.y
    )
    for trip_time in iso_min:
        subgraph_nodes = set()
        for center_node in center_nodes:
            nodes = nx.single_source_dijkstra_path_length(
                G_walk_projected, center_node, cutoff=trip_time, weight="time"
            )
            subgraph_nodes.update(nodes.keys())

        if len(subgraph_nodes) >= 3:
            x_coords = [G_walk_projected.nodes[node]["x"] for node in subgraph_nodes]
            y_coords = [G_walk_projected.nodes[node]["y"] for node in subgraph_nodes]
            poly = Polygon(zip(x_coords, y_coords)).convex_hull
            isochrone_polygons[trip_time].append(poly)

# Сливаем все полигоны изохрон в один для каждой временной зоны (и переводим в WGS84 для Folium)
iso_rows = []
for trip_time in iso_min:
    polys = isochrone_polygons[trip_time]
    if polys:
        temp_gdf = gpd.GeoDataFrame(geometry=polys, crs=G_walk_projected.graph["crs"])
        merged_poly = temp_gdf.union_all()
        iso_rows.append({"minutes": trip_time, "geometry": merged_poly})
gdf_isochrones = gpd.GeoDataFrame(iso_rows, crs=G_walk_projected.graph["crs"]).to_crs(
    epsg=4326
)

# Конвертируем ребра графа в GeoDataFrame (для последующей визуализации в Folium)
edges_gdf = ox.graph_to_gdfs(G_walk_projected, nodes=False, edges=True)

# Присваиваем серый цвет для всех пешеходных дорог
edges_gdf["road_color"] = "#9ecae1"


# СОЗДАЕМ ИНТЕРАКТИВНУЮ КАРТУ FOLIUM
# Базовая настройка подложки карты (с центром в границе района)
area_wgs84 = area_border.to_crs(epsg=4326)
m = folium.Map(
    location=[
        area_wgs84.geometry.centroid.y.iloc[0],
        area_wgs84.geometry.centroid.x.iloc[0],
    ],
    zoom_start=15,
    tiles="cartodbpositron",
)

# Настройка функции раскраски землепользования
def style_landuse(feature):
    props = feature["properties"]
    lu = props.get("landuse", props.get("leisure", "default"))

    # Если тип комплексный (например, парк, лес), приводим к базовому ключу (для корректного отображения на карте и в легенде)
    if lu in ["park", "garden", "grass", "forest"]:
        style = landuse_colors["park"]
    else:
        # Если тип землепользования не найден в словаре, используем ключ "default"
        style = landuse_colors.get(lu, landuse_colors   ["default"]) 
        
    return {
        # Настройка стиля для каждого типа землепользования (из библиотеки landuse_colors)
        "fillColor": style["fill"],
        "color": style["stroke"],
        "weight": 0.6,
        "fillOpacity": 0.6
    }

# Слой 0: Граница района
fg_area_boundary = folium.FeatureGroup(name="🛑 Граница района", show=True)
folium.GeoJson(
    area_wgs84,
    style_function=lambda x: {
        "fillColor": "none",
        "color": "#4a4a4a",
        "weight": 3,
        "dashArray": "8, 5",
    },
    tooltip="Граница исследуемого района",
).add_to(fg_area_boundary)
fg_area_boundary.add_to(m)

# Слой 1: Типы землепользования
fg_landuse = folium.FeatureGroup(name="🌳 Землепользование", show=True)
folium.GeoJson(
    landuse.to_crs(epsg=4326), style_function=style_landuse
).add_to(fg_landuse)
fg_landuse.add_to(m)

# Слой 2: Изохроны доступности
fg_isochrones = folium.FeatureGroup(name="⏱️ Зоны доступности метро", show=True)
for idx, row in gdf_isochrones.iterrows():
    t_min = row["minutes"]
    color = iso_colors[t_min]
    folium.GeoJson(
        row["geometry"],
        style_function=lambda x, col=color: {
            "fillColor": col,
            "color": col,
            "weight": 1.5,
            "fillOpacity": 0.25,
        },
        tooltip=f"Доступность: {t_min} минут пешком",
    ).add_to(fg_isochrones)
fg_isochrones.add_to(m)

# Слой 3: Пешеходная сеть
fg_network = folium.FeatureGroup(name="🛣️ Пешеходная сеть улиц", show=True)
folium.GeoJson(
    edges_gdf.to_crs(epsg=4326),
    style_function=lambda x: {
        "color": x["properties"]["road_color"],
        "weight": 1.2,
        "opacity": 0.45
    }
).add_to(fg_network)
fg_network.add_to(m)

# Слой 4: Застройка
fg_buildings = folium.FeatureGroup(name="🏢 Застройка (Здания)", show=True)
folium.GeoJson(
    buildings.to_crs(epsg=4326),
    style_function=lambda x: {
        "fillColor": "#737373",
        "color": "#252525",
        "weight": 0.5,
        "fillOpacity": 0.5,
    },
    # Настройка всплывающих подсказок для зданий (по типу)
    popup=folium.GeoJsonPopup(fields=["building"], aliases=["Тип здания:"])
    if "building" in buildings.columns
    else None,
).add_to(fg_buildings)
fg_buildings.add_to(m)

# Слой 5: Станции метрополитена
fg_metro = folium.FeatureGroup(name="🚇 Станции метро", show=True)
for idx, row in metro_stations.to_crs(epsg=4326).iterrows():
    name = row.get("name", "Неизвестно")
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=f"<b>Станция: {name}</b>",
        icon=folium.Icon(color="red", icon="subway", prefix="fa"),
    ).add_to(fg_metro)
fg_metro.add_to(m)

# Добавляем панель управления слоями (LayerControl)
folium.LayerControl(collapsed=False).add_to(m)

# Добавляем легенду карты с пояснениями по цветам изохрон и типам землепользования 
legend_html = f"""
<div style="
    position: fixed; 
    bottom: 50px; left: 50px; width: 260px; height: auto; 
    z-index:9999; font-size:12px; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    background-color:rgba(255, 255, 255, 0.92);
    box-shadow: 0 0 15px rgba(0,0,0,0.2);
    border-radius: 8px;
    padding: 15px;
    line-height: 18px;
    color: #333333;
">
    <b style="font-size:14px; display:block; margin-bottom:8px;">Легенда карты</b>
    
    <span style="font-weight:600; display:block; margin-top:5px; margin-bottom:3px; color:#555;">Доступность метро:</span>
    <i style="background:{iso_colors[5]}; width:16px; height:16px; float:left; margin-right:8px; opacity:0.6; border-radius:3px;"></i> до 5 минут<br>
    <i style="background:{iso_colors[10]}; width:16px; height:16px; float:left; margin-right:8px; opacity:0.6; border-radius:3px;"></i> 5–10 минут<br>
    <i style="background:{iso_colors[15]}; width:16px; height:16px; float:left; margin-right:8px; opacity:0.6; border-radius:3px;"></i> 10–15 минут<br>
    
    <span style="font-weight:600; display:block; margin-top:10px; margin-bottom:3px; color:#555;">Землепользование:</span>
    <i style="background:{landuse_colors['park']['fill']}; border:1px solid {landuse_colors['park']['stroke']}; width:16px; height:12px; float:left; margin-right:8px; opacity:0.7;"></i> {landuse_colors['park']['label']}<br>
    <i style="background:{landuse_colors['residential']['fill']}; border:1px solid {landuse_colors['residential']['stroke']}; width:16px; height:12px; float:left; margin-right:8px; opacity:0.7;"></i> {landuse_colors['residential']['label']}<br>
    <i style="background:{landuse_colors['industrial']['fill']}; border:1px solid {landuse_colors['industrial']['stroke']}; width:16px; height:12px; float:left; margin-right:8px; opacity:0.7;"></i> {landuse_colors['industrial']['label']}<br>
    <i style="background:{landuse_colors['allotments']['fill']}; border:1px solid {landuse_colors['allotments']['stroke']}; width:16px; height:12px; float:left; margin-right:8px; opacity:0.7;"></i> {landuse_colors['allotments']['label']}<br>
    <i style="background:{landuse_colors['cemetery']['fill']}; border:1px solid {landuse_colors['cemetery']['stroke']}; width:16px; height:12px; float:left; margin-right:8px; opacity:0.7;"></i> {landuse_colors['cemetery']['label']}<br>
    <i style="background:{landuse_colors['construction']['fill']}; border:1px solid {landuse_colors['construction']['stroke']}; width:16px; height:12px; float:left; margin-right:8px; opacity:0.7;"></i> {landuse_colors ['construction']['label']}<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# СОХРАНЕНИЕ КАРТЫ В ФАЙЛ
output_name = "index.html"
m.save(output_name)

/var/folders/rm/f1rv2xpn6d5dzrm6jv9pyywr0000gn/T/ipykernel_5161/1886442005.py:75: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  area_wgs84.geometry.centroid.y.iloc[0],
/var/folders/rm/f1rv2xpn6d5dzrm6jv9pyywr0000gn/T/ipykernel_5161/1886442005.py:76: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  area_wgs84.geometry.centroid.x.iloc[0],
